# SAR L2 Validation — Steps 4 & 5

This notebook demonstrates:
- **Step 4a** — SAR patch extraction (`collocation_patches.nc`)
- **Step 4b** — Validation statistics (bias, RMSE, correlation)
- **Step 5**  — Visualisation (scatter, geographic, statistics bar chart, residuals)

Prerequisites: steps 1–3 must already have been run (i.e. `datatree.nc` and
`collocation_results.nc` must exist in the data directory).

In [ ]:
import xarray as xr
from pathlib import Path

# ── Edit this path to match your data directory ──────────────────────────────
DATA_DIR = Path(
    "/home/chvan0015/git/sar-l2-validation-toolbox/data/"
    "2026-03-01-000000-2026-03-02-000000_-10.00_5.00_50.00_65.00"
)
RECIPE_PATH = Path("/home/chvan0015/git/sar-l2-validation-toolbox/recipes/test.yaml")
# ─────────────────────────────────────────────────────────────────────────────

print("Data directory:", DATA_DIR)
print("Files present:", [f.name for f in DATA_DIR.iterdir() if f.is_file()])

## Load DataTree and collocation results

In [ ]:
datatree = xr.open_datatree(str(DATA_DIR / "datatree.nc"), engine="netcdf4")
collocation_ds = xr.open_dataset(str(DATA_DIR / "collocation_results.nc"))

print("DataTree structure:")
print(datatree)
print("\nCollocation dataset:")
print(collocation_ds)

In [ ]:
# Inspect available variables
print("Collocation variables:")
for v in sorted(collocation_ds.data_vars):
    print(f"  {v}: {collocation_ds[v].dtype}")

## Load recipe

In [ ]:
from sar_validation.core.recipe import Recipe

recipe = Recipe.from_yaml(RECIPE_PATH)
print(f"Recipe: {recipe.config.name}")
print(f"Variable: {recipe.config.variable}")

## Step 4a — SAR patch extraction

Extract a 5×5 pixel neighbourhood from the SAR swath around each collocated point.
The result is saved to `collocation_patches.nc`.

In [ ]:
from sar_validation.core.patch_extractor import run_patch_extraction

PATCH_SIZE = 5  # pixels; must be odd

patches_ds = run_patch_extraction(
    collocation_ds=collocation_ds,
    datatree=datatree,
    patch_size=PATCH_SIZE,
    base_dir=DATA_DIR,
)

if patches_ds is not None:
    print(patches_ds)
    # Show the first patch for the first SAR variable
    patch_vars = [v for v in patches_ds.data_vars if v.startswith("sar_patch_")]
    if patch_vars:
        import matplotlib.pyplot as plt
        v = patch_vars[0]
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(patches_ds[v].isel(collocation=0).values, origin="upper", cmap="viridis")
        ax.set_title(f"{v}\n(first collocated point)")
        ax.set_xlabel("patch_x offset")
        ax.set_ylabel("patch_y offset")
        plt.tight_layout()
        plt.show()

## Step 4b — Validation statistics

In [ ]:
from sar_validation.core.statistics import run_statistics

stats_map = run_statistics(collocation_ds, recipe, DATA_DIR)

for key, ds in stats_map.items():
    print(f"\n── {key} ──")
    print(ds.to_dataframe())

## Step 5 — Visualisation

### 5a. Scatter plot

In [ ]:
from sar_validation.core.visualization import plot_scatter
from sar_validation.core._variable_map import infer_variable_pairs
import matplotlib.pyplot as plt

pairs = infer_variable_pairs(recipe.config.variable)

for sar_var, val_var in pairs:
    fig = plot_scatter(collocation_ds, sar_var, val_var, by_source=True)
    if fig:
        plt.show()

### 5b. Interactive scatter (requires plotly)

In [ ]:
try:
    for sar_var, val_var in pairs:
        fig = plot_scatter(collocation_ds, sar_var, val_var, interactive=True)
        if fig:
            fig.show()
except ImportError as e:
    print(f"Skipping interactive scatter: {e}")

### 5c. Geographic plot — one subplot per SAR scene

In [ ]:
from sar_validation.core.visualization import plot_geographic

for sar_var, val_var in pairs:
    try:
        fig = plot_geographic(datatree, collocation_ds, sar_var, ncols=2)
        if fig:
            plt.show()
    except Exception as e:
        print(f"Geographic plot failed for {sar_var}: {e}")

### 5d. Interactive geographic map (requires folium)

In [ ]:
try:
    for sar_var, val_var in pairs:
        m = plot_geographic(datatree, collocation_ds, sar_var, interactive=True)
        display(m)  # renders inline in Jupyter
except ImportError as e:
    print(f"Skipping interactive map: {e}")
except Exception as e:
    print(f"Interactive map failed: {e}")

### 5e. Statistics bar chart

In [ ]:
from sar_validation.core.visualization import plot_statistics

for key, stats_ds in stats_map.items():
    fig = plot_statistics(stats_ds, metrics=["bias", "rmse", "correlation"])
    if fig:
        plt.show()

### 5f. Residuals histogram

In [ ]:
from sar_validation.core.visualization import plot_residuals

for sar_var, val_var in pairs:
    fig = plot_residuals(collocation_ds, sar_var, val_var, by_source=True)
    if fig:
        plt.show()

### 5g. Full validation report

Run all plots at once and save PNGs to `<data_dir>/plots/`.

In [ ]:
from sar_validation.core.visualization import validation_report

figures = validation_report(
    collocation_ds=collocation_ds,
    datatree=datatree,
    recipe=recipe,
    stats_ds_map=stats_map,
    out_dir=DATA_DIR,
)

print("Saved PNG files:")
for f in sorted((DATA_DIR / "plots").glob("*.png")):
    print(f"  {f.name}")